In [1]:
import sys, os
sys.path.append("..")

from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

from src.runner import run_scenario
from src.database import get_connection

load_dotenv("../.env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4.1-mini"

In [2]:
scenario = "Netflix is raising the price of its standard plan by $3 per month."
scenario_id = run_scenario(client, MODEL, scenario, n_personas=20)

Scenario 1: saved 20/20 responses


In [3]:
query = """
SELECT
    p.income,
    r.decision,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY p.income), 1) AS pct
FROM responses r
JOIN personas p ON r.persona_id = p.persona_id
WHERE r.scenario_id = ?
GROUP BY p.income, r.decision
ORDER BY p.income, n DESC;
"""
conn = get_connection()
pd.read_sql_query(query, conn, params=(scenario_id,))

,income,decision,n,pct
0,$35k-$75k,accept,3,60.0
1,$35k-$75k,reduce,2,40.0
2,$75k-$150k,accept,3,50.0
3,$75k-$150k,reduce,3,50.0
4,over $150k,accept,1,50.0
5,over $150k,reduce,1,50.0
6,under $35k,reduce,4,57.1
7,under $35k,accept,3,42.9


In [4]:
query = """
SELECT
    p.price_sensitivity,
    COUNT(*) AS n,
    ROUND(AVG(r.intent_score), 2) AS avg_intent,
    SUM(CASE WHEN r.decision = 'accept' THEN 1 ELSE 0 END) AS n_accept
FROM responses r
JOIN personas p ON r.persona_id = p.persona_id
WHERE r.scenario_id = ?
GROUP BY p.price_sensitivity
ORDER BY CASE p.price_sensitivity
    WHEN 'low' THEN 1 WHEN 'medium' THEN 2 WHEN 'high' THEN 3 END;
"""
pd.read_sql_query(query, conn, params=(scenario_id,))

,price_sensitivity,n,avg_intent,n_accept
0,low,10,7.40,9
1,medium,7,6.00,1
2,high,3,5.67,0


In [5]:
scenario_2 = "Netflix is raising the price of its standard plan by $15 per month."
scenario_id_2 = run_scenario(client, MODEL, scenario_2, n_personas=20)


Scenario 2: saved 20/20 responses


In [6]:
query = """
SELECT
    s.description,
    r.decision,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY s.scenario_id), 1) AS pct
FROM responses r
JOIN scenarios s ON r.scenario_id = s.scenario_id
WHERE r.scenario_id IN (?, ?)
GROUP BY s.scenario_id, r.decision
ORDER BY s.scenario_id, n DESC;
"""
pd.read_sql_query(query, conn, params=(scenario_id, scenario_id_2))

,description,decision,n,pct
0,Netflix is raising the price of its standard p...,accept,10,50.0
1,Netflix is raising the price of its standard p...,reduce,10,50.0
2,Netflix is raising the price of its standard p...,reduce,12,60.0
3,Netflix is raising the price of its standard p...,reject,7,35.0
4,Netflix is raising the price of its standard p...,delay,1,5.0


In [7]:
pd.set_option("display.max_colwidth", None)